# host

> The application under an agent, and how it says what it can do.

In [ ]:
#| default_exp host

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import ast, json, os, re, uuid
from abc import ABC, abstractmethod
from pathlib import Path
from fastcore.basics import AttrDict, ifnone, patch
from fastcore.parallel import startthread
from shalya.core import Hit, HostError, MAX_API, MAX_FILE, MAX_GREP_HITS, host_err

## Saying what a host can do

A host is asked one question over and over: can you do this? Ramabana used to ask it five ways. A
`capabilities` dict. A check for whether the subclass overrode the method. A harmless call that
catches `NotImplementedError`. A signature inspection. Thirty lines of per-group special cases
saying which of the four applied where.

One way replaces all five. Each capability group is an abstract base class carrying the name of the
group and the methods it needs. A host declares a group by inheriting it, which is a decision rather
than a coincidence: `ABCMeta` refuses to build a host that inherits a group and leaves a method
unimplemented. `provides` reads the declarations off the class.

`without` is the second half, and it is why a class alone is not enough. Whether a host can search
the web is not a property of its class. It is a property of whether `fossick` imported and whether
the session was started with the web switched off. A host that inherits `WebHost` and finds no
`fossick` adds `web` to `without`, and answers honestly for the rest of its life.

In [ ]:
#| export
class Capability(ABC):
    "One capability group. A host declares the group by inheriting the class that names it."
    group = ''


The path boundary is not a capability. Every group needs it, so it is `Host` itself.

In [ ]:
#| export
class Host(ABC):
    "The application under an agent: the folders it may touch, and what it declares it can do."

    group = 'file'          #: every host has the path boundary the file tools need
    without = frozenset()   #: groups this instance cannot do, whatever its class declares

    @property
    def provides(self):
        "The groups this host actually supports: what its class declares, less `without`."
        declared = {c.group for c in type(self).__mro__ if getattr(c, 'group', '')}
        return declared - set(self.without)

    def can(self, group):
        "Whether this host supports one group."
        return group in self.provides

    @property
    @abstractmethod
    def roots(self):
        "The open folders, as absolute paths. The agent is told about these and confined to them."

    @property
    def added_roots(self):
        "The roots opened after this host was built, which a resumed session must not inherit."
        return []

    def add_root(self, path):
        "Open another folder, and return it resolved. Widening the write boundary, so hosts may refuse."
        raise HostError('this host cannot open another folder')

    @abstractmethod
    def check(self, path, must_exist=False, reading=False):
        """The single chokepoint: resolve `path`, refuse anything outside `roots`, return a `Path`.

        `reading=True` says the caller will only *read* what comes back, and it is the one case a
        host may answer for a path outside `roots`. See `LocalHost(read_outside=)`.
        """

    @abstractmethod
    def walk(self):
        "Every readable file under the open folders."

    @abstractmethod
    def read(self, path):
        "One file's text, or None when it cannot be read."

    @abstractmethod
    def write(self, path, text):
        "Write `text` to `path`, through the same sandbox `check` enforces. Returns the path written."

    @abstractmethod
    def text_at(self, path):
        "One file as a single diffable document, `''` when it does not exist yet, None on error."

    @property
    def approvals(self):
        "Where a write goes to be approved. Shalya never reads this. Ramabana and Leela both set it."
        return None

    def note(self, text):
        "Tell the user something out of band. Never blocks. A host may drop it."
        pass

## The groups

Nine of them. Each names its group and the methods that group's tools call, and nothing else: no
bodies, no state, no ordering between them. This is the whole contract between a host and the
toolset.

In [ ]:
#| export
class CodeHost(Capability):
    "Reading the shape of a codebase rather than its bytes."
    group = 'code'

    @abstractmethod
    def search(self, query, limit=20):
        "Search the code index for `query`, returning `Hit`s. Semantic if an index exists, literal if not."

    @abstractmethod
    def symbols(self, path):
        "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."

    @abstractmethod
    def peers(self, path, line, limit=20):
        "Code shaped like whatever is defined at `path`:`line`. Every place a pattern was already used."

    @abstractmethod
    def public_api(self, package, limit=MAX_API):
        """Every public name `package` exports, as `Hit`s whose `symbol` is the qualified name.

        Raise rather than return `[]` when there is no index. "This package exports nothing" and
        "nothing could be looked up" must not reach the model as the same answer.
        """

    def grep(self, pattern, path_filter='', regex=True, ignore_case=False, limit=MAX_GREP_HITS):
        "Every line matching `pattern` exactly, as `Hit`s. None means this host has no exact matcher."
        return None

    @property
    def indexed(self):
        "The folders whose index is built and searchable now. Empty means `public_api` has nothing to read."
        return ()

    @property
    def search_note(self):
        "Which engine answered, and why. Shown when a search finds nothing."
        return ''


class WebHost(Capability):
    "Going out to the network now, as opposed to recalling what was read before."
    group = 'web'

    @abstractmethod
    def web_search(self, query, n=20):
        "Search the web. Returns objects with `.title` and `.url`."

    @abstractmethod
    def read_url(self, url, remember=True):
        "One page as markdown. `remember=False` keeps sensitive or low-quality results ephemeral."

    @abstractmethod
    def research(self, query):
        "Search and read the top results into one cited digest. Slower than `web_search`."

    @property
    def research_note(self): return ''


class NotebookHost(Capability):
    """Notebooks, as far as a host has to know about them.

    Shalya owns no notebook representation. Exhash addresses cells by path and id without one, so
    only the two operations that need to know what a notebook *is* are delegated.
    """
    group = 'notebook'

    @abstractmethod
    def nb_cells(self, path):
        "`[(id, cell_type, source)]` for one notebook."

    @abstractmethod
    def nb_add_cell(self, path, source, index=-1, cell_type='code'):
        "Insert a cell (-1 appends), creating the notebook if needed. Returns the new cell's id."


class MemoryHost(Capability):
    "What was read before, kept as a tree rather than as a transcript."
    group = 'memory'

    @abstractmethod
    def memory_search(self, query, limit=8):
        "Search remembered pages as whole tree sections, returning structured rows."

    @abstractmethod
    def memory_tree(self, document=''):
        "The heading tree for remembered documents. An empty document lists every root."

    @abstractmethod
    def memory_read(self, node_id):
        "Read one remembered section and its children by stable node id."

    @abstractmethod
    def memory_topics(self, limit=12):
        "Labelled semantic clusters across remembered research."

    @abstractmethod
    def memory_forget(self, doc_id):
        "Purge one remembered document and all derived tree, chunk and vector data."

    @abstractmethod
    def remember(self, text, title=None, tags=()):
        "File `text` into durable memory as a note. Returns the document record."


class AskHost(Capability):
    """Answering out of memory, which takes a model.

    Its own group because it is a model call rather than a lookup. A host can remember pages and
    search them without having anything to ask them with.
    """
    group = 'ask'

    @abstractmethod
    def ask(self, question, ref=None, instruction='', **kw):
        "Answer `question` out of remembered research, with citations, as a dict."


class WatchHost(Capability):
    """What the agent arranged to read later.

    A watch is a job the host re-runs on an interval, `poll` is the tick, and a reminder is a watch
    that files its own text.
    """
    group = 'watch'

    @abstractmethod
    def watch(self, target, action='remind', every='1d', note=None, **params):
        "Register a recurring job. `target` is a URL, a query, or the text of a reminder."

    @abstractmethod
    def watches(self, due_only=False):
        "Every registered watch, soonest first. `due_only` keeps the ones that have come due."

    @abstractmethod
    def unwatch(self, watch_id):
        "Delete one watch. Whatever it already filed stays in memory."

    @abstractmethod
    def poll(self):
        "Run every watch that is due and report what fired. One failing watch must not stop the rest."

    @property
    def watch_actions(self):
        "The `action` values this host's `watch` will accept."
        return ('remind',)


class SessionHost(Capability):
    "The live namespace, and the terminal beside it."
    group = 'session'

    @abstractmethod
    def run_python(self, code):
        """Run `code` in the user's live namespace under whatever restrictions the host imposes.

        The contract the agent is briefed on, and the host's to enforce: read anything, bind
        results to new names, never rebind or delete the owner's.
        """

    @abstractmethod
    def inspect_python(self, code, scope='isolated'):
        """Run `code` against the live namespace without touching what the user has.

        Two scopes, both protecting the owner's variables, by different means:

        - `'isolated'` runs in an allowlist sandbox on a *copy*. Attribute reads and builtins
          work. Most library method calls are refused. The default, and it needs no trust.
        - `'overlay'` runs the real interpreter against the real namespace under an AST policy:
          read anything, bind names in the agent's own layer, never delete, rebind or mutate
          the owner's. `list(df.columns)` works here. In the sandbox it does not.

        A host may refuse `'overlay'`. See `scopes`.
        """

    @abstractmethod
    def list_vars(self):
        "What is in the live namespace: name, type, and a short value, one per line."

    def terminal_text(self, lines=200):
        "What the IDE's terminal has printed. Read-only: it shows what the user ran, it cannot run anything."
        return ''

    @property
    def scopes(self):
        "The scopes `inspect_python` will actually honour, most trusted last."
        return ('isolated',)

    @property
    def kernel_kind(self):
        "What runs the live namespace. `'ipymini'` inspects while a cell is busy. Anything else queues."
        return 'ipykernel'

    @property
    def concurrent(self): return self.kernel_kind == 'ipymini'


class ShellHost(Capability):
    "Running a command on the machine."
    group = 'shell'

    @abstractmethod
    def run_cmd(self, command, cwd=None, timeout=120):
        """Run `command` in a shell and return `(exit_code, combined_output)`.

        The contract a host must keep, because the tool trusts it:

        - `cwd` is resolved through `check`. Confining the *working directory* is not confining
          the command, which is why `run_shell` is a write tool and goes to a person.
        - stdout and stderr come back interleaved, in one string, in order.
        - `timeout` is enforced and the whole process *group* is killed on expiry.
        - A failed command returns a non-zero exit code rather than raising.
        """

    @property
    def shell_note(self):
        "How commands are run here, or why they are not."
        return ''


class ApiHost(Capability):
    "Reading an API specification and calling what it describes."
    group = 'api'

    @abstractmethod
    def api_load(self, src, name=''):
        "Load an OpenAPI specification from a path or a URL. Returns what it is now called."

    @abstractmethod
    def api_ops(self, group='', name='', match='', limit=None, offset=0):
        "The operations a loaded specification describes, filtered and paged."

    @abstractmethod
    def api_count(self, group='', name='', match=''):
        "How many operations that filter matches, without listing them."

    @abstractmethod
    def api_call(self, operation, name='', **params):
        "Call one operation from a loaded specification."


class GitHost(Capability):
    """A working tree the git tools can act on.

    The group needs no methods. `gheasy` reaches the repository through `roots`, so declaring the
    group is the whole contract.
    """
    group = 'git'

Nine of them, and each carries the name `tools_for` looks up. A group whose class and whose table
entry disagree is a group that silently never arrives.

In [ ]:
GROUP_CLASSES = (CodeHost, WebHost, NotebookHost, MemoryHost, AskHost, WatchHost, SessionHost,
                 ShellHost, ApiHost, GitHost)
sorted(c.group for c in GROUP_CLASSES)

In [ ]:
test_eq(sorted(c.group for c in GROUP_CLASSES),
        ['api', 'ask', 'code', 'git', 'memory', 'notebook', 'session', 'shell', 'watch', 'web'])
assert all(issubclass(c, Capability) for c in GROUP_CLASSES)
test_eq(Capability.group, '')                             # the base names no group
test_eq(Host.group, 'file')                               # and the path boundary is its own
test_eq(len({c.group for c in GROUP_CLASSES}), len(GROUP_CLASSES))   # no two share a name

`provides` is the answer to every "can you?" the toolset asks. Read it against a host that
declares two groups and has lost one of them.

In [ ]:
class Demo(Host, CodeHost, WebHost):
    without = {'web'}                                    # fossick is not installed here
    @property
    def roots(self): return ['/proj']
    def check(self, path, must_exist=False, reading=False): return Path('/proj')/path
    def walk(self): return []
    def read(self, path): return None
    def write(self, path, text): return str(path)
    def text_at(self, path): return ''
    def search(self, query, limit=20): return [Hit('/proj/a.py', 1, 'a', 'def a(): ...')]
    def symbols(self, path): return []
    def peers(self, path, line, limit=20): return []
    def public_api(self, package, limit=MAX_API): return []
    def web_search(self, query, n=20): return []
    def read_url(self, url, remember=True): return None
    def research(self, query): return ''

d = Demo()
sorted(d.provides), d.can('code'), d.can('web')

In [ ]:
test_eq(sorted(d.provides), ['code', 'file'])   # every host has the file group
test_eq(d.can('code'), True)
test_eq(d.can('web'), False)
test_eq(d.can('memory'), False)

Declaring a group and leaving a method out is refused where the host is built, naming the method.
This is what makes the declaration worth trusting: a host cannot claim a group it has not written.

In [ ]:
class Broken(Demo, NotebookHost):
    def nb_cells(self, path): return []                  # and no `nb_add_cell`

test_fail(Broken, contains='nb_add_cell')

## A host over real folders

`LocalHost` is the reference implementation: enough of a host to run an agent from a terminal, an
MCP server or a test. It declares every group, and drops the ones its optional dependencies or its
construction arguments say it cannot do. One class covers every combination of groups.
`mk_host(vault=True, spec=True)` sets two arguments where it used to synthesise a `VaultSpecHost` at
runtime. Adding a kernel to a running session becomes an assignment rather than a replacement.

In [ ]:
#| export
SANDBOX = 'path is outside the open folders'
SECRET = 'path holds credentials and is never read'
NO_ROOTS = 'no folders are open, so no path is inside them'

#: Credential-shaped paths refused even with `read_outside`. `fnmatch` on the resolved path.
DENY = ('*/.ssh/*', '*/.aws/*', '*/.gnupg/*', '*/.config/gcloud/*', '*/.netrc',
        '*/.git-credentials', '*/.codex/auth.json', '*/.claude/.credentials.json',
        '*/.env', '*/.env.*', '*/id_rsa*', '*/id_ed25519*', '*.pem', '*.key', '*.p12')
SKIP_DIRS = frozenset({'.git', '.hg', '.svn', '__pycache__', '.venv', 'venv', 'node_modules',
                       '.ipynb_checkpoints', '.pytest_cache', '.mypy_cache', '_docs', '_proc',
                       'dist', 'build', '.quarto', '.idea', '.attic'})
SKIP_SUFFIXES = frozenset({'.pyc', '.pyo', '.so', '.dylib', '.dll', '.a', '.o', '.zip', '.gz',
                           '.whl', '.png', '.jpg', '.jpeg', '.gif', '.webp', '.pdf', '.parquet',
                           '.sqlite', '.db', '.bin', '.safetensors', '.gguf'})
MAX_VARS = 200
LD_CHARS = 4000           # of a page's JSON-LD to keep. Enough for a product, not a catalogue

_LD = re.compile(r'<script[^>]+application/ld\+json[^>]*>(.*?)</script>', re.S | re.I)

The refusals and the skip lists are the host's, not a tool's. A tool that reported "outside the open
folders" in its own words would be a second place to change when the rule changes.

In [ ]:
SANDBOX, SECRET, NO_ROOTS

In [ ]:
assert all(isinstance(s, str) and s for s in (SANDBOX, SECRET, NO_ROOTS))
assert len(DENY) >= 15 and '*/.env' in DENY and '*.pem' in DENY
assert {'.git', '.venv', '__pycache__', 'node_modules'} <= SKIP_DIRS
assert {'.pyc', '.so', '.png', '.gguf'} <= SKIP_SUFFIXES
assert MAX_VARS > 0 and LD_CHARS > 0

In [ ]:
#| export
def denied(path, patterns=DENY):
    "Whether `path` is one of the things reading outside the open folders still must not open."
    from fnmatch import fnmatch
    s = Path(path).as_posix()
    return any(fnmatch(s, pat) for pat in patterns)

def _md_doc(d):
    "One of fossick's document readers' results as markdown: the fields it has, then its text."
    if isinstance(d, str): return d
    if not isinstance(d, dict): return str(d or '')
    head = [f'**{k}**: {v}' for k in ('title', 'authors', 'published', 'channel', 'duration', 'link')
            if (v := d.get(k)) not in (None, '', [], {})]
    body = next((str(d[k]) for k in ('source', 'text', 'content', 'summary') if d.get(k)), '')
    return '\n'.join(head + [''] + [body]).strip() if head else body.strip()

def _fuse(legs, limit):
    "Merge ranked `Hit` lists with `litesearch.rrf_all`. Identity is `path:line`."
    legs = [list(l) for l in legs if l]
    if not legs: return []
    if len(legs) == 1: return legs[0][:limit]
    from litesearch import rrf_all   # imported here: it pulls pandas, and only fusion needs it
    by_key, lists = {}, []
    for leg in legs:
        rows = []
        for h in leg:
            key = f'{h.path}:{h.line}'
            by_key.setdefault(key, h)
            rows.append({'_fid': key})
        lists.append(rows)
    try: fused = rrf_all(lists, id_key='_fid', limit=limit)
    except Exception: return legs[0][:limit]   # a bad fusion degrades the ranking, never `search`
    return [by_key[r['_fid']] for r in fused if r.get('_fid') in by_key]

def ld_json(html):
    "The `schema.org` JSON-LD blocks in `html`. Where a page states its price, author or rating."
    out = []
    for m in _LD.finditer(html or ''):
        try: out.append(json.loads(m.group(1)))
        except Exception: pass
    return out

`denied` is what `read_outside` still refuses: credential-shaped paths, matched by `fnmatch` on the
resolved path. A read tool is not a way to exfiltrate a key.

In [ ]:
[p for p in ('/home/k/.ssh/id_rsa', '/proj/.env', '/proj/key.pem', '/proj/app.py') if denied(p)]

In [ ]:
for p in ('/home/k/.ssh/id_rsa', '/home/k/.aws/credentials', '/proj/.env', '/proj/.env.local',
          '/proj/id_ed25519', '/proj/key.pem', '/proj/cert.p12', '/home/k/.netrc'):
    assert denied(p), p
for p in ('/proj/app.py', '/proj/README.md', '/proj/environment.yml'):
    assert not denied(p), p
test_eq(denied('/proj/.env', patterns=()), False)         # an empty deny list denies nothing

`ld_json` pulls the structured data out of a page, which is how a product or an article survives a
conversion to markdown that would otherwise throw its fields away.

In [ ]:
page = '<html><script type="application/ld+json">{"@type": "Product", "name": "a kettle"}</script></html>'
ld_json(page)

In [ ]:
test_eq(ld_json(page), [{'@type': 'Product', 'name': 'a kettle'}])
test_eq(ld_json('<html>nothing structured here</html>'), [])
test_eq(ld_json('<script type="application/ld+json">not json</script>'), [])
test_eq(ld_json(''), [])

Construction opens the folders and starts one Kosha sync per root in a daemon thread. It starts
before the first turn, so the index is usually ready by the time a search asks for it, and `search`
answers from ripgrep while it is not.

`without` is filled here, once, from what actually imported and what the caller asked for. Nothing
recomputes it later.

In [ ]:
#| export
class LocalHost(Host, CodeHost, WebHost, NotebookHost, SessionHost, ShellHost, MemoryHost,
                WatchHost, AskHost, ApiHost, GitHost):
    "The reference `Host`: enough of one to run an agent from a terminal, an MCP server or a test."

    def __init__(self,
                 roots=('.',),          # the folders the agent is confined to
                 ns=None,               # the live namespace. A fresh dict when None
                 approvals=None,        # an `Approvals`, or None to gate nothing
                 note=None,             # callable for out-of-band status lines
                 web=True,              # wire the web tools to fossick when it is installed
                 index=True,            # start a Kosha sync for every open root
                 graph=False,           # build Kosha's call graph during sync
                 rerank=True,           # reorder Kosha's hits with its flashrank cross-encoder
                 rerank_model=None,     # flashrank model name. None is its fast default
                 memory=None,           # a vishalakshi store, for the memory and watch groups
                 apis=None,             # whatever answers the api group, for the api group
                 kernel=None,           # a live kernel for the session group. None runs in process
                 read_outside=False,    # let read-only tools name any path on this machine
                 deny=DENY):            # what `read_outside` still refuses to open
        self._roots = [str(Path(r).expanduser().resolve()) for r in roots]
        self._added_roots = []         # opened later, and not inherited by a resumed session
        self.ns = ifnone(ns, {'__name__': '__main__'})
        self._approvals, self._note, self.web = approvals, note, web
        self.read_outside, self.deny = bool(read_outside), tuple(deny or ())
        self.transcript = []           # what this process has printed, for `read_terminal`
        self._indexes, self._index_errors, self._index_thread = [], [], None
        self._pending = list(self._roots)     # roots whose sync has not returned yet
        self.rerank, self.rerank_model, self._rerank_note = bool(rerank), rerank_model, ''
        self.graph, self.memory, self.apis = graph, memory, apis
        self.kernel = kernel
        self.without = self._absent()
        if index: self.sync_index()

    def _absent(self):
        "The groups this host cannot do: no backend attached, or the package never imported."
        out = set()
        if not self.web or not _installed('fossick'): out.add('web')
        if self.memory is None: out |= {'memory', 'watch', 'ask'}
        if self.apis is None: out.add('api')
        return frozenset(out)

#: Every method below arrives by `@patch`, which runs after `ABCMeta` has worked out what is
#: missing, so it cannot see any of them yet. `implemented(LocalHost)` at the foot of this notebook
#: asks again by the same rule, and anything nothing ever wrote goes back to being abstract.
LocalHost.__abstractmethods__ = frozenset()

In [ ]:
#| export
def _installed(mod):
    "Whether `mod` imports. Asked once per host, at construction, and never again."
    from importlib.util import find_spec
    try: return find_spec(mod) is not None
    except Exception: return False

A folder with a little source in it, and a host over it. `index=False` keeps Kosha out of the cells
that are about the boundary rather than about retrieval; one cell further down turns it on.

Everything below this point runs against that host.

In [ ]:
root = Path(tempfile.mkdtemp()).resolve()/'proj'
(root/'pkg').mkdir(parents=True)
(root/'pkg'/'sizes.py').write_text('def threshold(n):\n    "Half of n."\n    return n // 2\n')
(root/'pkg'/'use.py').write_text('from .sizes import threshold\n\ndef budget(): return threshold(8192)\n')
local = LocalHost([root], index=False)
sorted(local.provides), sorted(local.without)

In [ ]:
test_eq(sorted(local.provides), ['code', 'file', 'git', 'notebook', 'session', 'shell', 'web'])
test_eq(sorted(local.without), ['api', 'ask', 'memory', 'watch'])

### The path boundary

`check` is the reason this class exists. It resolves a path before comparing it, so `..` and a
symlink pointing out of the tree are both refused rather than followed.

In [ ]:
#| export
@patch(as_prop=True)
def roots(self:LocalHost): return list(self._roots)

The open folders, resolved and absolute. A relative path a caller gives is taken against the
first of them.

In [ ]:
local.roots

In [ ]:
test_eq(local.roots, [str(root)])
test_eq(len(local.roots), 1)

In [ ]:
#| export
@patch(as_prop=True)
def added_roots(self:LocalHost):
    "Roots opened after construction. `/resume` lapses these, and says that it did."
    return list(self._added_roots)

What `add_root` opened after construction. `/resume` lapses these rather than inheriting them,
because a folder someone opened mid-session is not part of what the session was started on.

In [ ]:
local.added_roots

In [ ]:
test_eq(local.added_roots, [])

In [ ]:
#| export
@patch
def add_root(self:LocalHost, path):
    """Open another folder for reading and writing, and index it. Returns it resolved.

    The one operation that widens the write boundary of a running session, so it refuses
    anything that is not already a directory rather than creating one.
    """
    p = Path(path).expanduser().resolve()
    if not p.exists(): raise HostError(f'no such folder: {p}')
    if not p.is_dir(): raise HostError(f'not a folder, so it cannot be a root: {p}')
    if str(p) in self._roots: return str(p)
    self._roots.append(str(p)); self._added_roots.append(str(p))
    self._pending.append(str(p))
    try: self.sync_index()
    except Exception as e: self._index_errors.append(host_err(e))
    return str(p)

The one operation that widens the write boundary of a running session, so it refuses anything
that is not already a directory rather than creating one.

In [ ]:
extra = Path(tempfile.mkdtemp()).resolve()/'extra'
extra.mkdir(parents=True)
(extra/'notes.md').write_text('read me\n')
local.add_root(str(extra)), local.added_roots

In [ ]:
test_eq(local.added_roots, [str(extra)])
test_eq(local.roots[-1], str(extra))                     # the new root is open for reads and writes
test_eq(local.add_root(str(extra)), str(extra))          # already open: no second entry
test_eq(len(local.added_roots), 1)
test_fail(lambda: local.add_root(str(extra/'nope')), contains='no such folder')
test_fail(lambda: local.add_root(str(extra/'notes.md')), contains='not a folder')

In [ ]:
#| export
@patch
def check(self:LocalHost, path, must_exist=False, reading=False):
    "Resolve `path`. Refuse outside `roots` (unless `read_outside` and `reading`). Walks stay confined."
    p = Path(path).expanduser()
    if not self._roots: raise HostError(f'{NO_ROOTS}: {p}')  # empty roots must refuse, not IndexError
    if not p.is_absolute(): p = Path(self._roots[0])/p
    p = p.resolve()  # collapse `..` and out-of-root symlinks before comparing
    if not any(p == Path(r) or Path(r) in p.parents for r in self._roots):
        if not (reading and self.read_outside): raise HostError(f'{SANDBOX}: {p}')
        if denied(p, self.deny): raise HostError(f'{SECRET}: {p}')
    if must_exist and not p.exists(): raise HostError(f'no such file: {p}')
    return p

The single chokepoint. It resolves before it compares, so `..` and a symlink pointing out of the
tree are refused rather than followed.

In [ ]:
local.check('pkg/sizes.py')

In [ ]:
test_fail(lambda: local.check('../../etc/passwd'), contains='outside the open folders')
test_fail(lambda: local.check('pkg/nope.py', must_exist=True), contains='no such file')
test_eq(local.check('pkg/sizes.py').name, 'sizes.py')
test_eq(local.check('pkg/nope.py').name, 'nope.py')      # a path to write need not exist yet

In [ ]:
#| export
@patch(as_prop=True)
def roots_note(self:LocalHost):
    "How paths are resolved here, in one line, for the briefing and a status bar."
    n = len(self._roots)
    return (f'{n} folder(s); reads may name any path on this machine, writes may not'
            if self.read_outside else f'{n} folder(s); nothing outside them is readable')

One line for a briefing and a status bar: how many folders, and whether a read may leave them.

In [ ]:
local.roots_note, LocalHost([root], index=False, read_outside=True).roots_note

In [ ]:
assert 'nothing outside them is readable' in local.roots_note
assert 'reads may name any path' in LocalHost([root], index=False, read_outside=True).roots_note

In [ ]:
#| export
@patch
def _walk(self:LocalHost, root):
    "Files under `root`, skipping the same generated dirs/suffixes `grep` covers."
    try:
        from rgapi import fd
        rows = fd(root=root, skip_dir=sorted(SKIP_DIRS), max_filesize=MAX_FILE,
                  exclude=[f'*{s}' for s in sorted(SKIP_SUFFIXES)])
        for p in rows:
            p = Path(p)
            if not p.is_absolute(): p = Path(root)/p
            if p.is_symlink() or not p.is_file(): continue
            yield p
        return
    except Exception: pass
    for p in sorted(Path(root).rglob('*')):
        if any(part in SKIP_DIRS for part in p.parts): continue
        if not p.is_file() or p.is_symlink(): continue
        if p.suffix.lower() in SKIP_SUFFIXES: continue
        try:
            if p.stat().st_size > MAX_FILE: continue
        except OSError: continue
        yield p

In [ ]:
#| export
@patch
def walk(self:LocalHost):
    return [p for r in self._roots for p in self._walk(r)]

Every readable file under the open folders, with the generated directories and binary suffixes
`SKIP_DIRS` and `SKIP_SUFFIXES` name left out.

In [ ]:
sorted(p.name for p in local.walk())

In [ ]:
got = {p.name for p in local.walk()}
assert {'sizes.py', 'use.py'} <= got, got
(root/'__pycache__').mkdir(exist_ok=True); (root/'__pycache__'/'x.pyc').write_text('junk')
assert 'x.pyc' not in {p.name for p in local.walk()}

In [ ]:
#| export
@patch
def read(self:LocalHost, path):
    try: return self.check(path, must_exist=True, reading=True).read_text(encoding='utf-8')
    except Exception: return None

One file's text, or None when it cannot be read. None rather than raising, because a tool asking
for a file that is not there is an ordinary turn.

In [ ]:
local.read('pkg/sizes.py'), local.read('pkg/nope.py')

In [ ]:
assert local.read('pkg/sizes.py').startswith('def threshold')
test_eq(local.read('pkg/nope.py'), None)
test_eq(local.read('/etc/passwd'), None)                 # outside the folders, so unreadable

In [ ]:
#| export
@patch
def write(self:LocalHost, path, text):
    p = self.check(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(str(text), encoding='utf-8')
    return str(p)

Through the same `check`, and it makes the parent directory. Returns the path written.

In [ ]:
local.write('pkg/new.py', 'x = 1\n')

In [ ]:
test_eq((root/'pkg'/'new.py').read_text(), 'x = 1\n')
test_eq(local.write('deep/er/f.py', 'y = 2\n'), str(root/'deep'/'er'/'f.py'))
test_fail(lambda: local.write('/etc/passwd', 'no'), contains='outside the open folders')

In [ ]:
#| export
@patch
def text_at(self:LocalHost, path):
    "One file as a diffable document: a notebook as its cell sources, anything else as text."
    try: p = self.check(path)
    except Exception: return None
    if not p.exists(): return ''
    if p.suffix == '.ipynb':
        try:
            from fastcore.nbio import read_nb
            return '\n\n'.join(''.join(c.source) for c in read_nb(p).cells)
        except Exception: return None
    try: return p.read_text(encoding='utf-8')
    except Exception: return None

One file as a single diffable document. `''` for a file that does not exist yet, so a diff against
a new file is a diff rather than an error, and a notebook diffs as its cell sources.

In [ ]:
local.text_at('pkg/sizes.py')[:30], local.text_at('pkg/nope.py')

In [ ]:
test_eq(local.text_at('pkg/nope.py'), '')
test_eq(local.text_at('pkg/sizes.py'), local.read('pkg/sizes.py'))

In [ ]:
#| export
@patch(as_prop=True)
def approvals(self:LocalHost): return self._approvals

Where a write goes to be approved. Shalya never reads it. Ramabana and Leela both set it.

In [ ]:
local.approvals, LocalHost([root], index=False, approvals='an Approvals').approvals

In [ ]:
test_eq(local.approvals, None)
test_eq(LocalHost([root], index=False, approvals='an Approvals').approvals, 'an Approvals')

In [ ]:
#| export
@patch
def note(self:LocalHost, text):
    self.transcript.append(str(text))
    if self._note:
        try: self._note(str(text))
        except Exception: pass

Out of band, and never blocking. Everything said is kept, which is what `read_terminal` reads back.

In [ ]:
said = []
noisy = LocalHost([root], index=False, note=said.append)
noisy.note('starting a kernel')
said, noisy.transcript

In [ ]:
test_eq(said, ['starting a kernel'])
test_eq(noisy.transcript, ['starting a kernel'])         # kept, and `terminal_text` reads it back
boom = LocalHost([root], index=False, note=lambda t: 1/0)
boom.note('swallowed')                                   # a broken note must not end a turn
test_eq(boom.transcript, ['swallowed'])

### Seeing the code

Three engines answer a search, and which one answered is part of the result. Kosha's hybrid index
when it has synced, ripgrep when it has not, and reading the files when neither is installed.

In [ ]:
#| export
@patch
def sync_index(self:LocalHost, wait=False, force=False):
    "Run `Kosha.sync` for every open root, once, in a daemon thread. Each root publishes as it returns."
    if self._index_thread is None or not self._index_thread.is_alive():
        def run():
            try:
                os.environ.setdefault('TQDM_DISABLE', '1')  # kosha's tqdm even with verbose=False
                from kosha import Kosha
            except Exception as e:
                self._index_errors.append(host_err(e)); self._pending = []; return
            for root in list(self._roots):
                try:
                    k = Kosha(dir=Path(root), busy_timeout=30000)
                    k.sync(dir=Path(root), verbose=False, force=force, pyproject=True, graph=self.graph)
                    self._indexes.append(k)
                except Exception as e: self._index_errors.append(host_err(e))
                finally:
                    try: self._pending.remove(root)
                    except ValueError: pass
        self._index_thread = startthread(run, daemon=True)
        self._index_thread.name = 'shalya-kosha-sync'
    if wait: self._index_thread.join()
    return self

One Kosha sync per root, in a daemon thread, started once. It runs at construction, so the index is
usually ready by the first search.

In [ ]:
quiet = LocalHost([root], index=False)
quiet.sync_index() is quiet, quiet._index_thread.name

In [ ]:
test_eq(quiet.sync_index(), quiet)                        # chainable, and starts nothing twice
test_eq(quiet._index_thread.name, 'shalya-kosha-sync')

In [ ]:
#| export
@patch(as_prop=True)
def index_ready(self:LocalHost):
    "Whether *every* open folder is indexed. `indexed` is the per-folder answer `search` uses."
    return bool(self._indexes) and not self._pending

Whether *every* open folder is indexed. `indexed` is the per-folder answer `search` actually uses.

In [ ]:
local.index_ready, local.indexed

In [ ]:
test_eq(local.index_ready, False)                         # this host was built with index=False
test_eq(local.indexed, [])

In [ ]:
#| export
@patch(as_prop=True)
def indexed(self:LocalHost):
    "The folders whose index is built and searchable now. The rest are still syncing."
    return [str(getattr(k, 'root', '')) for k in list(self._indexes)]

In [ ]:
#| export
@patch
def wait_index(self:LocalHost, timeout=None):
    "Wait for the automatic Kosha sync. Returns whether semantic search is ready."
    if self._index_thread is not None: self._index_thread.join(timeout)
    return self.index_ready

Joins the sync thread and answers whether semantic search is ready.

In [ ]:
local.wait_index(1), local.index_ready

In [ ]:
test_eq(local.wait_index(1), False)      # nothing was ever started on this host

In [ ]:
#| export
@patch
def _rg(self:LocalHost, query, limit, regex=False, ignore_case=False, path_filter='', per_file=5, every_file=False):
    "Search through `rgapi.rg`. `every_file=True` matches what `walk` yields, hidden files included."
    try: from rgapi import rg
    except Exception: return None
    pattern = query if regex else re.escape(query)
    kw = dict(case_sensitive=(False if ignore_case else None), smart_case=not ignore_case,
              max_filesize=MAX_FILE, timeout_ms=20_000)
    if path_filter: kw['glob'] = f'*{path_filter}*'
    if every_file:
        kw.update(hidden=True, ignore=False, skip_dir=sorted(SKIP_DIRS),
                  exclude=[f'*{s}' for s in sorted(SKIP_SUFFIXES)])
    hits, counts = [], {}
    try:
        for root in self._roots:
            # pull enough rows to honour per-file caps, then trim to `limit`
            pull = None if not per_file else max(limit * 8, limit)
            for m in rg(pattern, root=root, max_results=pull, **kw):
                if getattr(m, 'kind', 'match') != 'match': continue
                path = Path(m.path)
                if not path.is_absolute(): path = Path(root)/path
                path_s = str(path)
                if per_file:
                    n = counts.get(path_s, 0)
                    if n >= per_file: continue
                    counts[path_s] = n + 1
                hits.append(Hit(path_s, int(m.line_number), '', (m.line or '').strip()[:200]))
                if len(hits) >= limit: return hits
    except Exception: return None
    return hits

In [ ]:
#| export
@patch
def grep(self:LocalHost, pattern, path_filter='', regex=True, ignore_case=False, limit=MAX_GREP_HITS):
    "Exact matching through ripgrep. None when `rgapi` is unavailable. The tool then reads files itself."
    return self._rg(pattern, limit, regex=regex, ignore_case=ignore_case,
                    path_filter=path_filter, per_file=None, every_file=True)

Exact matching through ripgrep. None means this host has no exact matcher, and the tool then reads
the files itself rather than reporting no matches.

In [ ]:
local.grep('threshold')

In [ ]:
hits = local.grep('threshold')
assert hits and all(h.line >= 1 for h in hits), hits
assert any('sizes.py' in str(h.path) for h in hits)
test_eq(local.grep('no_such_symbol_anywhere'), [])
assert local.grep('def .*old', regex=True), 'a regex must reach ripgrep as a regex'

In [ ]:
#| export
@patch
def _ranked(self:LocalHost, call, **kw):
    "One Kosha context call, reordered by its cross-encoder when reranking is on and working."
    if self.rerank:
        try: return call(rerank=True, rerank_model=self.rerank_model, **kw)
        except Exception as e:   # flashrank fetches its model on first use. Fall back once
            self.rerank = False
            self._rerank_note = f'; reranking off ({host_err(e)})'
    return call(**kw)

In [ ]:
#| export
@patch
def _semantic(self:LocalHost, query, limit):
    "Kosha hybrid results (repo + env + graph) as the Host's stable `Hit` shape."
    indexes = list(self._indexes)
    if not indexes: return []
    out, seen = [], set()
    for k in indexes:
        try:
            rows = self._ranked(k.context, q=query, limit=limit, repo=True, env=True,
                                graph=self.graph, columns='content,metadata')
        except Exception as e:
            self._index_errors.append(host_err(e)); continue
        for row in rows:
            row = dict(row)
            meta = row.get('metadata') or {}
            if isinstance(meta, str):
                try: meta = ast.literal_eval(meta)
                except Exception: meta = {}
            path = str(meta.get('path') or row.get('path') or '')
            line = int(meta.get('lineno') or 1)
            key = (path, line)
            if key in seen: continue
            seen.add(key)
            symbol = meta.get('mod_name') or meta.get('name') or ''
            text = ' '.join(str(row.get('content') or '').split())[:240]
            out.append(Hit(path, line, str(symbol), text))
            if len(out) >= limit: return out
    return out

In [ ]:
#| export
@patch
def _scan(self:LocalHost, query, limit):
    "Every matching line, by reading the files. What is left when there is no index and no ripgrep."
    hits = []
    for p in self.walk():
        try: text = p.read_text(encoding='utf-8')
        except Exception: continue
        if query not in text: continue
        for i, line in enumerate(text.splitlines(), 1):
            if query in line:
                hits.append(Hit(str(p), i, '', line.strip()[:200]))
                if len(hits) >= limit: return hits
    return hits

In [ ]:
#| export
@patch
def search(self:LocalHost, query, limit=20):
    "The code index and the literal scan, fused by rank rather than tried in order."
    if not (query or '').strip(): return []
    rg = self._rg(query, limit)
    if (hits := _fuse([self._semantic(query, limit), rg or []], limit)): return hits
    return [] if rg is not None else self._scan(query, limit)

Kosha's index and ripgrep fused by rank rather than tried in order, falling back to reading the
files when neither is installed. An empty query answers `[]` rather than everything.

In [ ]:
local.search('threshold')[:2]

In [ ]:
assert local.search('threshold'), 'ripgrep found nothing'
test_eq(local.search(''), [])
test_eq(local.search('   '), [])

In [ ]:
#| export
@patch(as_prop=True)
def search_note(self:LocalHost):
    n, tot = len(self._indexes), len(self._roots)
    if n:
        where = f'{n} of {tot} folder(s)' if self._pending else f'{tot} folder(s)'
        return f'Kosha semantic + keyword index over {where} and environment fused with ripgrep{self._rerank_note}'
    if self._index_errors: return f'Kosha unavailable ({self._index_errors[-1]}); literal fallback'
    return 'Kosha sync in progress; literal fallback via ripgrep'

Which engine answered, and why. Part of the result: "no matches" and "nothing could be looked up"
must not reach a model as the same string.

This is the one cell here that builds a real index, so it is also where `wait_index`, `indexed` and
`public_api` are checked against one.

In [ ]:
indexing = LocalHost([root])
indexing.wait_index(180)
indexing.search_note

In [ ]:
assert 'fallback' in local.search_note, local.search_note      # never indexed
# the sync lands on its own thread, so read the note once and check that one reading
note = indexing.search_note
assert (note.startswith('Kosha semantic + keyword index over')
        or note.startswith('Kosha unavailable')
        or note == 'Kosha sync in progress; literal fallback via ripgrep'), note
if indexing.index_ready: test_eq(indexing.indexed, [str(root)])
assert indexing.search('threshold'), 'ripgrep answers whether or not the index has landed'

Which engine answered, and why. Part of the result: "no matches" and "nothing could be looked up"
must not reach a model as the same string.

In [ ]:
local.search_note

In [ ]:
#| export
@patch
def public_api(self:LocalHost, package, limit=MAX_API):
    "Kosha's public surface for `package`, `@patch`-added methods included."
    if not str(package or '').strip(): return []      # the capability probe
    indexes = list(self._indexes)
    if not indexes: raise HostError(f'no code index: {self.search_note}')
    out, seen = [], set()
    for k in indexes:
        try: rows = k.public_api(package, meta_cols='name,mod_name,docstring,path,lineno', limit=limit)
        except Exception as e: self._index_errors.append(host_err(e)); continue
        for row in rows:
            row = dict(row)
            name = str(row.get('mod_name') or row.get('name') or '')
            if not name or name in seen: continue
            seen.add(name)
            doc = ' '.join(str(row.get('docstring') or '').split())[:200]
            out.append(Hit(str(row.get('path') or ''), int(row.get('lineno') or 1), name, doc))
            if len(out) >= limit: return out
    return out

The one question a source file cannot answer: everything a package exports. It raises rather than
answering `[]` where there is no index, so the two cases stay distinguishable.

In [ ]:
test_fail(lambda: local.public_api('fastcore'), contains='no code index')

In [ ]:
test_eq(local.public_api(''), [])                        # the empty package is the harmless case
test_eq(local.indexed, [])
test_fail(lambda: local.public_api('fastcore'), contains='no code index')
# and with an index it answers rather than refusing, whatever the index happens to hold
if indexing.index_ready: assert isinstance(indexing.public_api('shalya'), list)

In [ ]:
#| export
@patch
def _defs(self:LocalHost, path):
    "Every def/class in one file as `(line, qualified_name, depth)`, by parsing rather than grepping."
    src = self.read(path)
    if src is None: return []
    try: tree = ast.parse(src)
    except SyntaxError: return []
    out = []
    def walk(node, prefix='', depth=0):
        for child in ast.iter_child_nodes(node):
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
                name = f'{prefix}{child.name}'
                out.append((child.lineno, name, depth))
                walk(child, f'{name}.', depth + 1)
    walk(tree)
    return out

In [ ]:
#| export
@patch
def symbols(self:LocalHost, path):
    "The defs and classes in one file, as `Hit`s whose `score` is the indent depth."
    p = self.check(path, reading=True)
    out = []
    for line, name, depth in self._defs(p):
        h = Hit(str(p), line, name, '')
        h.score = depth
        out.append(h)
    return out

From parsing rather than grepping, so a method is reported at its own depth under its qualified
name. `score` is the indent depth.

In [ ]:
[(h.symbol, h.line, h.score) for h in local.symbols('pkg/sizes.py')]

In [ ]:
test_eq([h.symbol for h in local.symbols('pkg/sizes.py')], ['threshold'])
test_eq(local.symbols('pkg/sizes.py')[0].score, 0)
(root/'pkg'/'cls.py').write_text('class A:\n    def m(self): pass\n')
test_eq([(h.symbol, h.score) for h in local.symbols('pkg/cls.py')], [('A', 0), ('A.m', 1)])
test_eq(local.symbols('pkg/nope.py'), [])

In [ ]:
#| export
@patch
def peers(self:LocalHost, path, line, limit=20):
    "Every other place the symbol defined at `path`:`line` is mentioned."
    p = self.check(path, reading=True)
    defs = self._defs(p)
    name = next((n for ln, n, _ in sorted(defs, key=lambda d: -d[0]) if ln <= int(line)), None)
    if name is None: return []
    leaf = name.split('.')[-1]
    return [h for h in self.search(leaf, limit * 2)
            if not (str(h.path) == str(p) and h.line == int(line))][:limit]

Where the symbol defined at a line is used. The useful call site rather than the definition, which
is why the defining line is filtered out of its own answer.

In [ ]:
local.peers('pkg/sizes.py', 1)

In [ ]:
peers = local.peers('pkg/sizes.py', 1)
assert any('use.py' in str(h.path) for h in peers), peers
assert not any(str(h.path) == str(root/'pkg'/'sizes.py') and h.line == 1 for h in peers)
assert local.peers('pkg/sizes.py', 999), 'the last def at or above the line is still a def'
test_eq(local.peers('pkg/sizes.py', 0), [])               # nothing is defined above line 1
test_eq(local.peers('pkg/nope.py', 1), [])                # nothing to be a peer of

### Notebooks, the session and the shell

In [ ]:
#| export
@patch
def nb_cells(self:LocalHost, path):
    from fastcore.nbio import read_nb
    nb = read_nb(self.check(path, must_exist=True, reading=True))
    return [(c.get('id', ''), c.cell_type, ''.join(c.source)) for c in nb.cells]

`(id, cell_type, source)` per cell, and nothing else. Shalya owns no notebook representation.

In [ ]:
local.write('nb/demo.ipynb', json.dumps(
    {'cells': [{'cell_type': 'code', 'id': 'c0', 'source': ['x = threshold(4096)'],
                'metadata': {}, 'outputs': [], 'execution_count': None}],
     'metadata': {}, 'nbformat': 4, 'nbformat_minor': 5}))
local.nb_cells('nb/demo.ipynb')

In [ ]:
test_eq(len(local.nb_cells('nb/demo.ipynb')), 1)
test_eq(local.nb_cells('nb/demo.ipynb')[0][:2], ('c0', 'code'))
assert 'threshold(4096)' in local.nb_cells('nb/demo.ipynb')[0][2]
test_fail(lambda: local.nb_cells('nb/nope.ipynb'), contains='no such file')

In [ ]:
#| export
@patch
def nb_add_cell(self:LocalHost, path, source, index=-1, cell_type='code'):
    from fastcore.nbio import read_nb, write_nb, mk_cell, dict2nb
    p = self.check(path)
    nb = read_nb(p) if p.exists() else dict2nb({'cells': [], 'metadata': {}, 'nbformat': 4, 'nbformat_minor': 5})
    cell = mk_cell(source, cell_type)
    if not cell.get('id'): cell['id'] = uuid.uuid4().hex[:8]
    nb.cells.append(cell) if index < 0 else nb.cells.insert(int(index), cell)
    p.parent.mkdir(parents=True, exist_ok=True)
    write_nb(nb, p)
    return cell['id']

Writes the notebook when it does not exist, and returns the new cell's id. `-1` appends.

In [ ]:
cid = local.nb_add_cell('nb/new.ipynb', 'y = 1')          # writes the notebook
second = local.nb_add_cell('nb/new.ipynb', '# a heading', cell_type='markdown')
first = local.nb_add_cell('nb/new.ipynb', 'z = 0', index=0)
[c[:2] for c in local.nb_cells('nb/new.ipynb')]

In [ ]:
ids = [c[0] for c in local.nb_cells('nb/new.ipynb')]
test_eq(ids, [first, cid, second])                       # index=0 went first, -1 appended
test_eq([c[1] for c in local.nb_cells('nb/new.ipynb')], ['code', 'code', 'markdown'])
assert 'a heading' in local.text_at('nb/new.ipynb')      # and a notebook diffs as its sources

In [ ]:
#| export
@patch
def _exec(self:LocalHost, code, ns):
    "Run `code` in `ns`, returning printed output plus the last expression's value."
    import contextlib, io
    buf = io.StringIO()
    tree = ast.parse(str(code))
    last = tree.body.pop() if tree.body and isinstance(tree.body[-1], ast.Expr) else None
    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
        if tree.body: exec(compile(tree, '<agent>', 'exec'), ns)
        value = eval(compile(ast.Expression(last.value), '<agent>', 'eval'), ns) if last else None
    out = buf.getvalue()
    if value is not None: out += ('' if not out or out.endswith('\n') else '\n') + repr(value)
    return out.strip() or '(no output)'

In [ ]:
#| export
@patch
def run_python(self:LocalHost, code):
    "Run `code` in the live namespace. Failures come back as text: a tool cannot usefully raise."
    if self.kernel is not None: return self.kernel.run(code)
    try: return self._exec(code, self.ns)
    except Exception as e: return f'{host_err(e)}'

The live namespace persists between calls, which is what makes "bind results to new names" a rule
worth briefing an agent on. A failure comes back as text: a tool that raises ends the turn.

In [ ]:
local.run_python('import math\nradii = [1, 2, 3]'), local.run_python('areas = [math.pi*r*r for r in radii]\nlen(areas)')

In [ ]:
test_eq(local.run_python('len(areas)'), '3')
test_eq(local.run_python('print("a side effect")'), 'a side effect')
assert 'NameError' in local.run_python('no_such_name + 1')
test_eq(local.run_python(''), '(no output)')

In [ ]:
#| export
@patch
def inspect_python(self:LocalHost, code, scope='isolated'):
    "Run `code` without rebinding anything the user made. A kernel enforces this; a copy approximates it."
    if scope not in self.scopes: return f'this host only honours {self.scopes}'
    if self.kernel is not None: return self.kernel.inspect(code, scope)
    try: return self._exec(code, dict(self.ns))
    except Exception as e: return f'{host_err(e)}'

Against a copy, so a name it binds is gone when it returns. The copy is shallow, which is the whole
of what this host protects: a binding cannot move, an object can.

In [ ]:
local.inspect_python('doubled = [r*2 for r in radii]\nlen(doubled)'), local.run_python('"doubled" in globals()')

In [ ]:
test_eq(local.inspect_python('doubled = [r*2 for r in radii]\nlen(doubled)'), '3')
test_eq(local.run_python('"doubled" in globals()'), 'False')   # the binding never landed
local.inspect_python('radii.append(99)')
test_eq(local.run_python('len(radii)'), '4')                   # the object did
assert 'only honours' in local.inspect_python('1+1', scope='overlay')

In [ ]:
#| export
@patch(as_prop=True)
def scopes(self:LocalHost):
    "What `inspect_python` honours. A kernel says for itself; in process it is the shallow copy alone."
    return self.kernel.scopes if self.kernel is not None else ('isolated',)

What `inspect_python` honours. A kernel says for itself; in process it is the shallow copy alone.

In [ ]:
local.scopes

In [ ]:
test_eq(local.scopes, ('isolated',))       # no kernel, so the shallow copy alone

In [ ]:
#| export
@patch(as_prop=True)
def kernel_kind(self:LocalHost):
    return self.kernel.kind if self.kernel is not None else 'inprocess'

What runs the live namespace. `concurrent` is the one thing the harness needs from it: whether the
kernel can be inspected while a cell is busy, or whether a call has to queue.

In [ ]:
local.kernel_kind, local.concurrent

In [ ]:
test_eq(local.kernel_kind, 'inprocess')
test_eq(local.concurrent, False)

In [ ]:
#| export
@patch
def list_vars(self:LocalHost):
    if self.kernel is not None: return self.kernel.list_vars()
    rows = []
    for k, v in list(self.ns.items())[:MAX_VARS]:
        if k.startswith('_') or callable(v) or isinstance(v, type(ast)): continue
        try: short = repr(v)
        except Exception: short = '<unreprable>'
        rows.append(f'{k:20} {type(v).__name__:12} {short[:60]}')
    return '\n'.join(rows)

Name, type and a short value, one line each. Callables and dunders are left out: an agent asking
what is in the namespace wants the data, not the imports.

In [ ]:
print(local.list_vars())

In [ ]:
rows = dict(l.split(None, 1) for l in local.list_vars().splitlines())
assert 'radii' in rows and 'list' in rows['radii'], rows
assert 'math' not in rows and not any(k.startswith('_') for k in rows)

In [ ]:
#| export
@patch
def terminal_text(self:LocalHost, lines=200):
    "What this process has printed, when the application records it in `transcript`."
    return '\n'.join(str(x) for x in self.transcript[-int(lines):])

What this process has printed, when the application records it. Read-only: it shows what was run,
it cannot run anything.

In [ ]:
chatty = LocalHost([root], index=False)
for i in range(3): chatty.note(f'line {i}')
chatty.terminal_text(2)

In [ ]:
test_eq(chatty.terminal_text(2), 'line 1\nline 2')
test_eq(chatty.terminal_text(), 'line 0\nline 1\nline 2')
test_eq(LocalHost([root], index=False).terminal_text(), '')

In [ ]:
#| export
@patch
def run_cmd(self:LocalHost, command, cwd=None, timeout=120):
    """Run `command` in a shell under one of the open folders.

    Started in its own process group, and the *group* is killed on timeout. A command
    that spawns children cannot leave one behind. Stdout and stderr are interleaved.
    """
    import subprocess
    if not str(command or '').strip(): return 0, ''   # the capability probe
    if not (cwd or self._roots): raise HostError(NO_ROOTS)
    d = self.check(cwd) if cwd else Path(self._roots[0])
    if not d.is_dir(): raise HostError(f'not a directory: {d}')
    p = subprocess.Popen(str(command), shell=True, cwd=str(d), text=True, errors='replace',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         start_new_session=True)
    try: out, _ = p.communicate(timeout=max(1, int(timeout)))
    except subprocess.TimeoutExpired:
        import os, signal
        try: os.killpg(p.pid, signal.SIGKILL)
        except Exception: p.kill()
        out, _ = p.communicate()
        return 124, (out or '') + f'\n[killed after {int(timeout)}s]'
    return p.returncode, out or ''

In its own process group under the first open folder, with stdout and stderr interleaved. A
failure is an exit code rather than an exception, and the group is killed on timeout.

In [ ]:
local.run_cmd('echo hi'), local.run_cmd('exit 3')

In [ ]:
test_eq(local.run_cmd('echo hi'), (0, 'hi\n'))
test_eq(local.run_cmd('exit 3')[0], 3)
test_eq(local.run_cmd('pwd')[1].strip(), str(root))
test_eq(local.run_cmd('echo out; echo err 1>&2')[1], 'out\nerr\n')   # interleaved, in order
test_eq(local.run_cmd(''), (0, ''))
code, out = local.run_cmd('sleep 5', timeout=1)
test_eq(code, 124); assert 'killed after 1s' in out

In [ ]:
#| export
@patch(as_prop=True)
def shell_note(self:LocalHost):
    return f'shell, in {self._roots[0]}' if self._roots else 'no folder to run a command in'

How commands are run here, or why they are not.

In [ ]:
local.shell_note

In [ ]:
assert str(root) in local.shell_note

### The web

Every page arrives as markdown. A URL that is a paper, a repository file or a video gets the reader
that knows its shape. Everything else is fetched. A page that comes back too thin to be real is
fetched again the expensive way.

In [ ]:
#| export
@patch
def _fossick(self:LocalHost):
    if not self.web: raise NotImplementedError
    try:
        import fossick
        return fossick
    except Exception: raise NotImplementedError

In [ ]:
#| export
@patch
def web_search(self:LocalHost, query, n=20):
    "Search the web through fossick. An empty query answers `[]`: that is how `tools_for` probes."
    fossick = self._fossick()
    if not str(query).strip(): return []
    rows = fossick.search(str(query), n=int(n))   # `n` to fossick. Its own default is 10
    return [AttrDict(title=str(r.get('title', '')), url=str(r.get('href') or r.get('url', ''))) for r in rows]

Through fossick, and an empty query answers `[]` without reaching the network.

In [ ]:
local.can('web'), local.web_search('')

In [ ]:
test_eq(local.web_search(''), [])                        # the probe must not hit the network
offline = LocalHost([root], index=False, web=False)
test_eq(offline.can('web'), False)
test_fail(lambda: offline.web_search('anything'), NotImplementedError)

In [ ]:
#| export
@patch
def read_url(self:LocalHost, url, remember=True):
    "Page as markdown via fossick: `READERS`, then `fetch(auto=True)`, thin-page escalate, JSON-LD."
    fossick = self._fossick()
    for rx, name, kw in self.READERS:
        if not rx.search(str(url)) or (reader := getattr(fossick, name, None)) is None: continue
        try: text = _md_doc(reader(str(url), **kw))
        except Exception as e:   # a reader that cannot answer is not a URL that cannot be read
            self.note(f'{name} could not read {url} ({host_err(e)}); fetching the page')
            break
        if text.strip(): return AttrDict(text=text, url=str(url))
        break
    page = fossick.fetch(str(url), auto=True)
    text = str(fossick.to_md(page) or '') if page is not None else ''
    if len(text.strip()) < self.THIN_PAGE:
        for opts in ({'heavy': True}, {'stealthy': True}):
            try: heavy = fossick.fetch(str(url), **opts)
            except Exception: continue
            if len((got := str(fossick.to_md(heavy) or '')).strip()) >= self.THIN_PAGE:
                page, text = heavy, got
                break
    if (ld := ld_json(getattr(page, 'html_content', '') or '')):
        text = f'<structured-data>\n{json.dumps(ld)[:LD_CHARS]}\n</structured-data>\n\n{text}'
    return None if not text.strip() else AttrDict(text=text, url=str(url))

In [ ]:
#| export
@patch
def research(self:LocalHost, query):
    "The cited corpus fossick assembled: its `digest`, and not the record it assembled it from."
    return str((self._fossick().research(str(query)) or {}).get('digest') or '')

In [ ]:
#| export
@patch(as_prop=True)
def research_note(self:LocalHost): return 'fossick' if self.web else 'web access is switched off'

LocalHost.THIN_PAGE = 400
LocalHost.READERS = (
        (re.compile(r'https?://(www\.)?github\.com/[^/]+/[^/]+/(blob|raw)/', re.I), 'read_gh_file', {}),
        (re.compile(r'https?://(www\.)?arxiv\.org/(abs|pdf)/', re.I), 'read_arxiv', dict(save_pdf=False, source=True)),
        (re.compile(r'https?://(www\.)?(youtube\.com/watch|youtu\.be/)', re.I), 'read_yt', {}),
    )

Which engine reads the web here, or why none does.

In [ ]:
local.research_note, LocalHost([root], index=False, web=False).research_note

In [ ]:
test_eq(local.research_note, 'fossick')
test_eq(LocalHost([root], index=False, web=False).research_note, 'web access is switched off')

In [ ]:
#| export
def _needs(host, what):
    "The refusal a forwarding method gives when its backend was never attached."
    return HostError(f'this host has no {what}')

In [ ]:
#| export
@patch
def memory_search(self:LocalHost, query, limit=8):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.search(str(query), limit=int(limit))

In [ ]:
#| export
@patch
def memory_tree(self:LocalHost, document=''):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.tree(str(document or ''))

In [ ]:
#| export
@patch
def memory_read(self:LocalHost, node_id):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.read(str(node_id))

In [ ]:
#| export
@patch
def memory_topics(self:LocalHost, limit=12):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.topics(limit=int(limit))

In [ ]:
#| export
@patch
def memory_forget(self:LocalHost, doc_id):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.forget(str(doc_id))

In [ ]:
#| export
@patch
def remember(self:LocalHost, text, title=None, tags=()):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.remember(str(text), title=title, tags=tuple(tags))

In [ ]:
#| export
@patch
def ask(self:LocalHost, question, ref=None, instruction='', **kw):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.ask(str(question), ref=ref, instruction=instruction, **kw)

In [ ]:
#| export
@patch
def watch(self:LocalHost, target, action='remind', every='1d', note=None, **params):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.watch(target, action=action, every=every, note=note, **params)

In [ ]:
#| export
@patch
def watches(self:LocalHost, due_only=False):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.watches(due_only=bool(due_only))

In [ ]:
#| export
@patch
def unwatch(self:LocalHost, watch_id): 
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.unwatch(str(watch_id))

In [ ]:
#| export
@patch
def poll(self:LocalHost):
    if self.memory is None: raise _needs(self, 'vault')
    return self.memory.poll()

### Memory, watches and an API specification

Three groups this host does not implement itself. It holds the backend and forwards to it, and says
so through `without` when there is no backend to forward to. The memory store is a
[vishalakshi](https://github.com/vedicreader/vishalakshi) vault, opened by whatever built the host.

In [ ]:
class Store:
    "The surface `LocalHost` forwards the memory and watch groups to."
    def __init__(self): self.docs, self.jobs, self.n = {}, {}, 0
    def remember(self, text, title=None, tags=()):
        self.n += 1
        doc = {'id': f'd{self.n}', 'title': title or text.split('\n')[0][:40], 'text': text, 'tags': list(tags)}
        self.docs[doc['id']] = doc
        return doc
    def search(self, query, limit=8):
        return [d for d in self.docs.values() if query.lower() in d['text'].lower()][:limit]
    def tree(self, document=''):
        return ([{'id': d['id'], 'title': d['title']} for d in self.docs.values()] if not document
                else [{'id': document, 'title': self.docs[document]['title']}])
    def read(self, node_id): return self.docs[node_id]
    def topics(self, limit=12): return [{'topic': t, 'n': 1} for d in self.docs.values() for t in d['tags']][:limit]
    def forget(self, doc_id): return self.docs.pop(doc_id, None) is not None
    def ask(self, question, ref=None, instruction='', **kw):
        hits = self.search(question)
        return {'answer': hits[0]['text'] if hits else '(nothing remembered)',
                'cited': [{'n': 1, 'breadcrumb': h['title'], 'node_id': h['id']} for h in hits]}
    def watch(self, target, action='remind', every='1d', note=None, **params):
        self.n += 1
        self.jobs[f'w{self.n}'] = {'id': f'w{self.n}', 'target': target, 'action': action, 'every': every}
        return self.jobs[f'w{self.n}']
    def watches(self, due_only=False): return [w for w in self.jobs.values() if not due_only]
    def unwatch(self, watch_id): return self.jobs.pop(watch_id, None) is not None
    def poll(self): return [{'id': w['id'], 'fired': True} for w in self.jobs.values()]

store = Store()
remembering = LocalHost([root], index=False, memory=store)
sorted(remembering.provides), sorted(remembering.without)

In [ ]:
test_eq(sorted(remembering.without), ['api'])              # a store answers memory, ask and watch
assert remembering.can('memory') and remembering.can('ask') and remembering.can('watch')
test_eq(local.can('memory'), False)

Each forward is one line, and calling into a group with no backend says which backend is missing
rather than raising an `AttributeError` from inside the forward.

In [ ]:
doc = remembering.remember('the kettle boils at 100C', title='kettles', tags=['home'])
remembering.memory_search('kettle'), remembering.memory_tree(), remembering.memory_read(doc['id'])['title']

In [ ]:
test_eq(remembering.memory_search('kettle')[0]['id'], doc['id'])
test_eq(remembering.memory_search('nothing like this'), [])
test_eq(remembering.memory_tree(), [{'id': doc['id'], 'title': 'kettles'}])
test_eq(remembering.memory_read(doc['id'])['text'], 'the kettle boils at 100C')
test_eq(remembering.memory_topics(), [{'topic': 'home', 'n': 1}])
test_eq(remembering.ask('kettle')['answer'], 'the kettle boils at 100C')
test_eq(remembering.memory_forget(doc['id']), True)
test_eq(remembering.memory_tree(), [])                    # and it is really gone
# a host with no store says which backend is missing, rather than raising from inside the forward
for nm in ('memory_tree', 'memory_search', 'ask', 'remember', 'memory_topics', 'memory_read',
           'memory_forget'):
    try: getattr(local, nm)('x')
    except HostError as e: assert 'no vault' in str(e), (nm, e)
    else: assert False, f'{nm} did not refuse'

A watch is a job the host re-runs on an interval, and `poll` is the tick.

In [ ]:
w = remembering.watch('https://example.com/changelog', action='remind', every='1d')
remembering.watches(), remembering.watch_actions, remembering.poll()

In [ ]:
test_eq(remembering.watches(), [w])
test_eq(remembering.watches(due_only=True), [])
test_eq(remembering.watch_actions, ('remind',))
test_eq(remembering.poll(), [{'id': w['id'], 'fired': True}])
test_eq(remembering.unwatch(w['id']), True)
test_eq(remembering.watches(), [])
for call in (lambda: local.watch('x'), lambda: local.watches(), lambda: local.unwatch('w1'),
             lambda: local.poll()):
    test_fail(call, contains='no vault')

In [ ]:
#| export
@patch
def api_load(self:LocalHost, src, name=''):
    if self.apis is None: raise _needs(self, 'API specifications')
    return self.apis.api_load(src, name=name)

In [ ]:
#| export
@patch
def api_ops(self:LocalHost, group='', name='', match='', limit=None, offset=0):
    if self.apis is None: raise _needs(self, 'API specifications')
    return self.apis.api_ops(group=group, name=name, match=match, limit=limit, offset=offset)

In [ ]:
#| export
@patch
def api_count(self:LocalHost, group='', name='', match=''):
    if self.apis is None: raise _needs(self, 'API specifications')
    return self.apis.api_count(group=group, name=name, match=match)

In [ ]:
#| export
@patch
def api_call(self:LocalHost, operation, name='', **params):
    if self.apis is None: raise _needs(self, 'API specifications')
    return self.apis.api_call(operation, name=name, **params)

The api group forwards the same way. A backend here is anything answering the four calls, which is
what `ramabana.spec.SpecHost` is.

In [ ]:
class Apis:
    "The surface `LocalHost` forwards the api group to."
    def __init__(self): self.ops = []
    def api_load(self, src, name=''):
        self.ops = [{'name': 'listPets', 'group': 'pets'}, {'name': 'getPet', 'group': 'pets'}]
        return {'name': name or 'petstore', 'operations': len(self.ops)}
    def api_ops(self, group='', name='', match='', limit=None, offset=0):
        return [o for o in self.ops if (not group or o['group'] == group) and (not match or match in o['name'])]
    def api_count(self, group='', name='', match=''): return len(self.api_ops(group=group, match=match))
    def api_call(self, operation, name='', **params): return {'called': operation, 'with': params}

speccy = LocalHost([root], index=False, apis=Apis())
speccy.api_load('https://example.com/openapi.json'), speccy.api_ops(match='list')

In [ ]:
test_eq(speccy.can('api'), True)
test_eq(speccy.api_load('x')['operations'], 2)
test_eq(speccy.api_count(), 2)
test_eq([o['name'] for o in speccy.api_ops(match='list')], ['listPets'])
test_eq(speccy.api_ops(group='nothing'), [])
test_eq(speccy.api_call('getPet', id=7), {'called': 'getPet', 'with': {'id': 7}})
for call in (lambda: local.api_load('x'), lambda: local.api_ops(), lambda: local.api_count(),
             lambda: local.api_call('x')):
    test_fail(call, contains='no API specifications')

`@patch` adds a method after the class is built, and `ABCMeta` worked out what was still missing
while it was building. `implemented` asks the question again, by the same rule ABCMeta uses. A
method that nothing ever wrote stays abstract, so a host with a hole in it still refuses to be
built.

In [ ]:
#| export
def implemented(cls):
    "Recompute what `cls` is still missing, after `@patch` filled some of it in."
    names = {m for base in cls.__mro__ for m in getattr(base, '__abstractmethods__', ())}
    cls.__abstractmethods__ = frozenset(
        n for n in names if getattr(getattr(cls, n, None), '__isabstractmethod__', False))
    return cls

implemented(LocalHost)

`implemented` is what makes the `@patch` style above safe. A method nothing ever wrote stays
abstract, so a host with a hole in it still refuses to be built.

In [ ]:
class Hollow(Host, ApiHost):
    "Declares the api group and writes none of it."
    @property
    def roots(self): return []
    def check(self, path, must_exist=False, reading=False): return Path(path)
    def walk(self): return []
    def read(self, path): return None
    def write(self, path, text): return str(path)
    def text_at(self, path): return ''

Hollow.__abstractmethods__ = frozenset()      # as the LocalHost cell does, to allow the patches
test_fail(implemented(Hollow), contains='api_load')

In [ ]:
@patch
def api_load(self:Hollow, src, name=''): return {}
@patch
def api_ops(self:Hollow, group='', name='', match='', limit=None, offset=0): return []
@patch
def api_count(self:Hollow, group='', name='', match=''): return 0
@patch
def api_call(self:Hollow, operation, name='', **params): return {}

implemented(Hollow)
test_eq(Hollow().provides, {'api', 'file'})   # written, so it builds, and it declares what it wrote
test_eq(Hollow.__abstractmethods__, frozenset())

### A kernel, attached to a host that already exists

`run_python` and its neighbours ask `self.kernel` when there is one. That is the whole of what a
frontend needs to move a session onto a real kernel: one assignment on the host it already has.

Ramabana used to build a replacement host from four attributes of the old one, and every group the
old host had and `LocalHost` does not went with it. For a session started with a vault and an API
spec that was fifteen tools, gained none. Nothing is copied here, so nothing can be dropped.

In [ ]:
class Kernel:
    "The shape `LocalHost` expects of a kernel: four calls and two facts."
    scopes, kind = ('isolated', 'overlay'), 'ipykernel'
    def __init__(self): self.ns = {}
    def run(self, code): exec(code, self.ns); return 'ran on the kernel'
    def inspect(self, code, scope='isolated'): return f'inspected under {scope}'
    def list_vars(self): return ', '.join(k for k in self.ns if not k.startswith('__'))

k = LocalHost([root], index=False)
before = sorted(k.provides)
k.kernel = Kernel()
before, sorted(k.provides), k.scopes, k.kernel_kind

In [ ]:
test_eq(sorted(k.provides), before)                 # attaching a kernel takes nothing away
test_eq(k.scopes, ('isolated', 'overlay'))
test_eq(k.kernel_kind, 'ipykernel')
test_eq(k.run_python('answer = 42'), 'ran on the kernel')
test_eq(k.list_vars(), 'answer')
test_eq(k.inspect_python('answer', 'overlay'), 'inspected under overlay')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()